In [ ]:
from delta.tables import DeltaTable
import json

env = dbutils.widgets.get("env").lower()
jobRunId = dbutils.widgets.get("jobRunId")
taskRunId = dbutils.widgets.get("taskRunId")
workflowId = dbutils.widgets.get("workflowId")

In [ ]:


control = (
    spark.table("master_control")
    .filter(f"workflowId = '{workflowId}' AND layer = 'GOLD'")
    .first()
)

cfg = json.loads(control.colMapping)

source_query = cfg["sourceQuery"]

target_table = control.targetTable
load_type = control.loadType

# -----------------------------
# READ SILVER USING QUERY
# -----------------------------
df = spark.sql(source_query)

# -----------------------------
# WRITE GOLD
# -----------------------------
if load_type.upper() == "OVERWRITE":

    df.write.format("delta").mode("overwrite").saveAsTable(target_table)

elif load_type.upper() == "APPEND":

    df.write.format("delta").mode("append").saveAsTable(target_table)

elif load_type.upper() == "MERGE":

    # assume customerID exists in gold
    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(df.alias("s"), "t.customerID = s.customerID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )